## Download the taxi data

In [1]:
import urllib.request #loads Python’s built-in tool for downloading files from the internet.
from pathlib import Path #loads a tool for working with file and folder paths.

DATA = Path("../data") #points to your data folder. .. means “go up one folder,” because the notebook is inside notebooks.
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet" #is the web address of January 2025 NYC Yellow Taxi trips, published by the city.
file = DATA / "yellow_tripdata_2025-01.parquet" #is where the file will be saved: inside data, with that name.

urllib.request.urlretrieve(url, file) #downloads the file from url and saves it to
print("Downloaded:" , file) #shows a message when it’s finished.

Downloaded: ..\data\yellow_tripdata_2025-01.parquet


## Load the data and take a first look 

In [2]:
import pandas as pd #loads pandas, the data table library. as pd is a short nickname everyone uses, so you type pd instead of pandas

trips = pd.read_parquet(file) #reads the downloaded file into a table and stores it in a variable called trips. (file still holds the path from the previous cell.)
print(trips.shape) #shows the table’s size as (rows, columns). Each row is one taxi trip.
trips.head() #shows the first 5 rows, so you can see what a trip looks like. It’s the last line, so Jupyter displays it as a neat table automatically.

(3475226, 20)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


## Get summary statistics 

In [3]:
trips.describe().T #describe() calculates statistics for every number column: count, average (mean), smallest (min), largest (max), and the percentiles in between (25%, 50%, 75%).
#.T flips the table sideways (T for “transpose”), so each column becomes a row. That makes it much easier to read.

,count,mean,min,25%,50%,75%,max,std
VendorID,3475226.0,1.785428,1.0,2.0,2.0,2.0,7.0,0.426328
tpep_pickup_datetime,3475226,2025-01-17 11:02:55.910964,2024-12-31 20:47:55,2025-01-10 07:59:01,2025-01-17 15:41:33,2025-01-24 19:34:06,2025-02-01 00:00:44,NaN
tpep_dropoff_datetime,3475226,2025-01-17 11:17:56.997901,2024-12-18 07:52:40,2025-01-10 08:15:29.500000,2025-01-17 15:59:34,2025-01-24 19:48:31,2025-02-01 23:44:11,NaN
passenger_count,2935077.0,1.297859,0.0,1.0,1.0,1.0,9.0,0.75075
trip_distance,3475226.0,5.855126,0.0,0.98,1.67,3.1,276423.57,564.6016
RatecodeID,2935077.0,2.482535,1.0,1.0,1.0,1.0,99.0,11.632772
PULocationID,3475226.0,165.191576,1.0,132.0,162.0,234.0,265.0,64.529483
DOLocationID,3475226.0,164.125177,1.0,113.0,162.0,234.0,265.0,69.401686
payment_type,3475226.0,1.036623,0.0,1.0,1.0,1.0,5.0,0.701333
fare_amount,3475226.0,17.081803,-900.0,8.6,12.11,19.5,863372.12,463.472918


**NOTE;** negative money values: usually refunds, voided trips or disputes being reserved. They are not real trips so they dont belong in the context of our analysis or the problem we are trying to solve/detect, "normal fare analysis". Huge maximums: a 276,000-mile trip is longer than driving to the Moon. That’s a broken meter or a data entry error.

## Count how many trips are affected 

In [4]:
print("Negative fares:" , (trips["fare_amount"] < 0).sum())
print("Trips over 100 miles:" , (trips["trip_distance"] > 100).sum())
print("Total trips:" , len(trips))

Negative fares: 144118
Trips over 100 miles: 162
Total trips: 3475226


**NOTE;** Over 100 miles: 162 out of 3.5 million is 0.005%. Dropping them is the right call.

Negative fares: 144,118 is about 4% of all trips. That’s not tiny. They should be dropped from the “normal trips” table, because our expected-fare model must learn from real, normal trips. But don’t throw them away. Refunds and disputes can be a fraud signal, so we’ll keep them in a separate table to study later.

**The lesson:** in a fraud project, “remove from analysis” and “delete forever” are different things.

## Split into clean and removed trips 

In [5]:
bad = (trips["fare_amount"] <0) | (trips["trip_distance"] >100)

removed = trips[bad]
clean = trips[~bad]

print("Clean trips:" , len(clean))
print("Removed trips:" , len(removed))

Clean trips: 3330970
Removed trips: 144256


## Calculate trip duration and speed

The data tells us when each trip started and ended, but not how long it took or how fast the taxi went. We’ll create those columns ourselves. This is called feature engineering, and it’s how we’ll catch impossible trips.

In [12]:
clean = clean.copy() #makes clean its own independent table. Without it, pandas shows a warning when we add columns, because clean was cut out of trips
clean["duration_min"] = (clean["tpep_dropoff_datetime"] - clean["tpep_pickup_datetime"]).dt.total_seconds() / 60 #creates a new column. Dropoff time minus pickup time gives the trip length, .dt.total_seconds() turns it into seconds, and / 60 turns seconds into minutes.
clean["speed_mph"] = clean["trip_distance"] / (clean["duration_min"] / 60) #clean["speed_mph"] = ... creates a speed column: miles divided by hours. Dividing minutes by 60 gives hours.
clean[["duration_min" , "speed_mph"]].describe() #shows summary statistics for just these two new columns.

C:\Users\super\nyc-taxi-integrity\.venv\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,duration_min,speed_mph
count,3.330970e+06,3.330286e+06
mean,1.503453e+01,inf
std,3.927097e+01,NaN
min,-5.147232e+04,-6.436800e+04
25%,7.266667e+00,7.226277e+00
50%,1.166667e+01,9.514286e+00
75%,1.831667e+01,1.290820e+01
max,5.626317e+03,inf
